# Fundamentals of Computer Vision

This Jupyter Notebook covers **Project 3** of the course and focuses on implementing Harris Corner Detector and LoG Blob Detector to introduce fundamental concepts in image processing. Each section has questions that must be answered in a Document in PDF format.

**Important**

Both the Harris Corner Detector and LoG Blob Detector tasks (Code + Answers) must be submitted; otherwise, your work will be rejected.


## Grading Breakdown: ##
- Harris Corner Detector: 50 points (Code: 30 pts, Answers: 20 pts).
- LoG Blob Detector: 50 points (Code: 30 pts, Answers: 20 pts).

To pass Project 3, a minimum of **50 points** is required.

# **Feature Extraction**

## **Harris Corner Detector**

In this exercise, you will implement the Harris corner detection algorithm to identify corner features in images. Follow these steps to complete the implementation of the `harris_corner_detection()` function:

1. Convert to Grayscale: Convert the input image to grayscale with floating-point precision.

2. Compute Image Gradients: Compute the first derivatives $I_x$ and $I_y$ using Sobel filter, which was implemented in Project 1.

3. Compute Gradient Products: Calculate $I_x^2$, $I_y^2$ and $I_x I_y$.

4. Smooth with Gaussian: Smooth the gradient products using a Gaussian filter.

5. Construct Local Structure Matrix (M): Form the local structure matrix $M$ using the smoothed gradient products:

\begin{equation}
M = \begin{bmatrix}
I_x^2 & I_x I_y \\
I_x I_y & I_y^2
\end{bmatrix}
\end{equation}

6. Calculate Eigenvalues: Compute the eigenvalues $\lambda_1$ and $\lambda_2$ of the matrix $M$.

7. Compute Harris Response: Calculate the Harris response $R = \lambda_1 \lambda_2 - k (\lambda_1 + \lambda_2)^2$.

8. Threshold and Mark Corners: Threshold the response $R$ to detect corners and highlight them in the image.

9. Return: Return the image with corners marked in red and Harris response $R$.

In [ ]:
# Import libraries
import cv2
import numpy as np
import matplotlib.pyplot as plt

def harris_corner_detector(image, k=0.04, window_size=3, threshold=1e-2):

    # Convert to grayscale
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    gray = gray.astype(np.float32) / 255.0

    # Compute Image Gradients
    Ix = cv2.Sobel(gray, cv2.CV_32F, 1, 0, ksize=3)
    Iy = cv2.Sobel(gray, cv2.CV_32F, 0, 1, ksize=3)

    # Compute Gradient Products
    Ix2 = Ix * Ix
    Iy2 = Iy * Iy
    Ixy = Ix * Iy

    # Smooth with Gaussian filter
    if window_size % 2 == 0:
        window_size += 1
    Ix2 = cv2.GaussianBlur(Ix2, (window_size, window_size), 0)
    Iy2 = cv2.GaussianBlur(Iy2, (window_size, window_size), 0)
    Ixy = cv2.GaussianBlur(Ixy, (window_size, window_size), 0)

    # Construct Local Structure Matrix
    a = Ix2
    b = Ixy
    c = Iy2

    # Calculate Eigenvalues
    trace = a + c
    det = a * c - b * b
    discriminant = np.sqrt(np.maximum(trace * trace - 4.0 * det, 0.0))
    lambda1 = (trace + discriminant) / 2.0
    lambda2 = (trace - discriminant) / 2.0

    # Compute Harris response
    R = lambda1 * lambda2 - k * (lambda1 + lambda2) ** 2

    # Threshold and mark corners
    R_max = R.max()
    if R_max != 0:
        R_norm = R / R_max
    else:
        R_norm = R

    corner_mask = R_norm > threshold

    output_image = image.copy()
    output_image[corner_mask] = [0, 0, 255]



    return output_image, R

# Load the image
path_images = "images/"
image = cv2.imread(path_images + "lenna.png")


# Detect corners
harris_image, response = harris_corner_detector(image)

# Display the result
plt.imshow(cv2.cvtColor(harris_image, cv2.COLOR_BGR2RGB))
plt.title('Harris Corner Detection')
plt.show()

**Questions that must be included in your Report**.

Answer the questions briefly and directly, including the images obtained from the execution of your code to explain your conclusions. Answers must be connected to the executed code.


1. Why is Gaussian smoothing applied to the gradient products $I_x^2$, $I_y^2$ and $I_x I_y$, and what effect does it have on the results?

2. What would happen if the matrix $M$ were structured as:

\begin{equation}
M = \begin{bmatrix}
I_x^2 & 0 \\
0 & I_y^2
\end{bmatrix}
\end{equation}

3. How does the Harris response $R$ reflect the strength of a corner, and what would happen if the threshold value was set too high or too low?

4. What is the significance of the constant $k$ in the Harris response formula, and how would changing its value influence the detection of corners?

5. The Harris Corner Detector:

  A. Is it robust in detecting corners, even in the presence of noise?

  B. Is it not invariant to scale or rotation?

  C. Is it sensitive to changes in illumination?

  D. Does it fail to inherently handle affine transformations, limiting its applicability for tracking features across different perspectives?


# **LoG Blob Detector**

In this exercise, you will implement the Laplacian of Gaussian (LoG)algorithm to identify blobs in images. Follow these steps to complete the implementation of the `detect_blobs_log()` function:

1. Generate LoG Filters: Create scale-normalized Laplacian of Gaussian filters at several scales (different scales means different sigma).

2. Convolve Image: Apply the LoG filters to the image and build a scale-space representation.

3. Find Peaks: Find maxima of squared Laplacian response in scale-space and apply a threshold to filter weak responses.

4. Draw Blobs: Visualise detected blobs with circles corresponding to their scale.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import maximum_filter

def detect_blobs_log(image, scales, threshold=0.1):
    # Convert to grayscale
    if len(image.shape) == 3:
        gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    else:
        gray = image.copy()

    # Convert to float64 for processing
    gray = gray.astype(np.float64) / 255.0

    # Generate LoG Filters
    scale_space = []

    for sigma in scales:
        # Apply Gaussian blur
        blurred = cv2.GaussianBlur(gray, (0, 0), sigma)

        # Apply Laplacian operator
        laplacian = cv2.Laplacian(blurred, cv2.CV_64F)

        # Scale normalization
        laplacian_normalized = (sigma ** 2) * laplacian

        # we take absolute value to detect both dark and bright blobs (negative Laplacian = dark blob, positive = bright blob)
        laplacian_abs = np.abs(laplacian_normalized)

        scale_space.append(laplacian_abs)

    # Convert scale_space to 3D numpy array
    scale_space = np.array(scale_space)

    # Find Peaks
    blobs = []

    # Find local maxima and also check for maxima across scales
    for scale_idx, sigma in enumerate(scales):
        # Get the response at this scale
        response = scale_space[scale_idx]

        response_thresholded = response * (response > threshold)

        # Find local maxima
        neighborhood_size = 3
        local_max = maximum_filter(response_thresholded, size=neighborhood_size)

        # Points that are local maxima in spatial domain
        is_max_spatial = (response_thresholded == local_max) & (response_thresholded > 0)

        # Check if it's also a maximum across scales
        for y, x in zip(*np.where(is_max_spatial)):
            # Check neighboring scales
            is_max_scale = True

            if scale_idx > 0:
                if scale_space[scale_idx - 1, y, x] > response[y, x]:
                    is_max_scale = False

            if scale_idx < len(scales) - 1:
                if scale_space[scale_idx + 1, y, x] > response[y, x]:
                    is_max_scale = False

            if is_max_scale:
                blobs.append((x, y, sigma, response[y, x]))

    # Draw Blobs
    if len(image.shape) == 2:
        output = cv2.cvtColor(image, cv2.COLOR_GRAY2BGR)
    else:
        output = image.copy()

    # Draw circles for each detected blob
    for x, y, sigma, response in blobs:
        radius = int(np.sqrt(2) * sigma)
        cv2.circle(output, (x, y), radius, (0, 255, 0), 2)
        # Draw center point
        cv2.circle(output, (x, y), 2, (0, 0, 255), -1)

    # Display results
    plt.figure(figsize=(15, 5))

    plt.subplot(1, 3, 1)
    if len(image.shape) == 3:
        plt.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
    else:
        plt.imshow(image, cmap='gray')
    plt.title('Original Image')
    plt.axis('off')

    plt.subplot(1, 3, 2)
    plt.imshow(cv2.cvtColor(output, cv2.COLOR_BGR2RGB))
    plt.title(f'Detected Blobs (count: {len(blobs)})')
    plt.axis('off')

    plt.subplot(1, 3, 3)
    # Show the maximum response across scales
    max_response = np.max(scale_space, axis=0)
    plt.imshow(max_response, cmap='hot')
    plt.title('Max LoG Response Across Scales')
    plt.colorbar()
    plt.axis('off')

    plt.tight_layout()
    plt.show()

    print(f"Detected {len(blobs)} blobs")
    return blobs


# Load the image
image = cv2.imread(path_images + 'butterfly.jpg') #, cv2.IMREAD_GRAYSCALE)

# Define the scales (sigma values)
scales = [1.5, 1.7, 2, 6]  # You can adjust these for more resolution at different scales

threshold = 0.1

# Detect blobs using LoG
detect_blobs_log(image, scales, threshold)


**Questions that must be included in your Report**.

Answer the questions briefly and directly, including the images obtained from the execution of your code to explain your conclusions. Answers must be connected to the executed code.

6. What is the purpose of the Laplacian of Gaussian (LoG) filter in the context of blob detection?

7. How does the sigma value affect the LoG filter, and why is it important to use multiple scales when detecting blobs?

8. LoG Blob Detector:

  A. Is the LoG blob detector robust to noise?

  B. Is the LoG blob detector scale-invariant but not rotation-invariant?

  C. How does the LoG blob detector handle changes in illumination?

  D. Does the LoG blob detector fail to inherently handle affine transformations, limiting its applicability for tracking features across different perspectives?

